# Analyze results from LLM Annotation of Applied Prognostic Evidence Line Comments

LLM metadata
- model_id: us.anthropic.claude-sonnet-4-6
- temperature: 0.0

In [1]:
import math
from pathlib import Path

import polars as pl

In [2]:
# This file includes LLM and human annotations for analysis
SAMPLE_PATH = Path(
    "public-sample_prognostic_annotated_ingested_vafs_2026-04-28-run2-temp0.0.tsv"
)

## Helper functions

In [3]:
def confusion_matrix_with_collapse(
    df: pl.DataFrame,
    gt_col: str,
    pred_col: str,
    collapse: bool = False,
) -> pl.DataFrame:
    """Compute a confusion matrix for:
    ['poor outcome', 'better outcome', 'unclear', 'null'],
    with optional merging of 'unclear' and 'null' into 'indeterminate'.

    :param df: Input DataFrame with evidence lines, associated comments, curator annotations, and LLM annotations
    :param gt_col: ground truth/human curator values column name (e.g., 'Curator Annotation')
    :param pred_col: predicted column name (e.g., 'LLM Annotation')
    :param collapse: whether to collapse 'unclear' and 'null' into 'indeterminate'
    :return: confusion matrix as a DataFrame
    """
    # Normalize ground truth
    gt = (
        pl.col(gt_col)
        .fill_null("null")
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
    )
    # Normalize predictions
    pred = (
        pl.col(pred_col)
        .fill_null("null")
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
    )

    working_df = df.with_columns(
        [
            gt.alias("gt"),
            pred.alias("pred"),
        ]
    )

    if collapse:
        collapse_map = {
            "unclear": "indeterminate",
            "null": "indeterminate",
        }

        working_df = working_df.with_columns(
            [
                pl.col("gt").replace(collapse_map),
                pl.col("pred").replace(collapse_map),
            ]
        )

        classes = [
            "poor outcome",
            "better outcome",
            "indeterminate",
        ]

    else:
        classes = [
            "poor outcome",
            "better outcome",
            "unclear",
            "null",
        ]

    cm = (
        working_df.group_by(["gt", "pred"])
        .len()
        .pivot(
            values="len",
            index="gt",
            on="pred",
        )
        .fill_null(0)
    )

    # Ensure all rows exist
    missing_rows = [cls for cls in classes if cls not in cm["gt"].to_list()]

    if missing_rows:
        cm = cm.with_columns(pl.col("gt").cast(pl.Utf8))

        cm = pl.concat(
            [
                cm,
                pl.DataFrame(
                    {
                        "gt": pl.Series(missing_rows, dtype=pl.Utf8),
                        **{
                            c: pl.Series(
                                [0] * len(missing_rows),
                                dtype=cm.schema.get(c, pl.UInt32),
                            )
                            for c in classes
                        },
                    }
                ),
            ],
            how="diagonal",
        )

    # Ensure all columns exist
    for cls in classes:
        if cls not in cm.columns:
            cm = cm.with_columns(pl.lit(0).alias(cls))

    # Reorder
    cm = cm.select(["gt", *classes])

    # Sort rows
    cm = (
        cm.with_columns(
            pl.col("gt").replace({v: i for i, v in enumerate(classes)}).alias("_order")
        )
        .sort("_order")
        .drop("_order")
    )

    return cm.rename({"gt": "Ground Truth"})

In [4]:
def create_analysis_summary(cm: pl.DataFrame) -> pl.DataFrame:
    """Compute per-class recall-style summary from a confusion matrix.

    :param cm: Confusion matrix (square DataFrame)
    :return: Summary DataFrame per class
    """
    rows = []

    classes = cm.columns[1:]

    for cls in classes:
        row = cm.filter(pl.col("Ground Truth") == cls)

        if row.height == 0:
            numerator = 0
            denominator = 0
        else:
            numerator = row.select(pl.col(cls)).item()

            denominator = row.select(pl.sum_horizontal(classes)).item()

        rows.append(
            {
                "consensus_w_curator": cls,
                "numerator": numerator,
                "denominator": denominator,
                "percentage": (
                    (numerator or 0) / denominator * 100
                    if denominator is not None and denominator > 0
                    else 0.0
                ),
            }
        )

    return pl.DataFrame(rows)

In [5]:
def compute_overall_accuracy(cm: pl.DataFrame) -> float:
    """Compute overall accuracy from a confusion matrix.

    :param cm: Confusion matrix DataFrame (square matrix)
    :return: Accuracy as a float between 0 and 1
    """
    classes = cm.columns[1:]

    total = cm.select(pl.sum_horizontal(classes)).to_series().sum()

    correct = 0

    for cls in classes:
        row = cm.filter(pl.col("Ground Truth") == cls)

        if row.height > 0:
            correct += row.select(pl.col(cls)).item() or 0

    return correct / total if total > 0 else math.nan

In [6]:
def analyze_results(
    df: pl.DataFrame,
    collapse: bool = True,
) -> tuple[pl.DataFrame, float, pl.DataFrame]:
    """Evaluate LLM predictions against ground truth using a confusion matrix.

    Computes:
      - confusion matrix (optionally collapsed classes)
      - per-class performance summary (recall-style)
      - overall accuracy

    :param df: Input DataFrame with ground truth and predictions
    :param collapse: If True, merges 'unclear' and 'null' into 'indeterminate'
    :return:
        - match_analysis_summary: per-class recall summary
        - accuracy: overall accuracy
        - cm: confusion matrix
    """
    analysis_df = df.clone()

    cm = confusion_matrix_with_collapse(
        analysis_df,
        "Curator Annotation",
        "LLM Annotation",
        collapse=collapse,
    )

    match_analysis_summary = create_analysis_summary(cm)

    accuracy = compute_overall_accuracy(cm)

    return match_analysis_summary, accuracy, cm

In [7]:
def process_results(
    collapse: bool,
) -> tuple[dict, dict, list]:
    """Run analysis on a stored model output

    :param collapse:  Whether to collapse categories during analysis
    :return: Tuple of all results (summaries, cm, accuracies)
    """
    _df = pl.read_csv(SAMPLE_PATH, separator="\t")

    summary, accuracy, cm = analyze_results(
        _df,
        collapse=collapse,
    )

    return (
        summary,
        cm,
        accuracy,
    )

## Analysis of Runs

### Main paper

In [8]:
summary, cm, accuracy = process_results(collapse=True)

In [9]:
cm

Ground Truth,poor outcome,better outcome,indeterminate
str,u32,u32,u32
"""poor outcome""",49,1,2
"""better outcome""",0,50,0
"""indeterminate""",25,7,26


In [10]:
summary

consensus_w_curator,numerator,denominator,percentage
str,i64,i64,f64
"""poor outcome""",49,52,94.230769
"""better outcome""",50,50,100.0
"""indeterminate""",26,58,44.827586


In [11]:
accuracy

0.78125

### Supplemental

In [12]:
sup_summary, sup_cm, sup_accuracy = process_results(collapse=False)

In [13]:
sup_cm

Ground Truth,poor outcome,better outcome,unclear,null
str,u32,u32,u32,u32
"""poor outcome""",49,1,1,1
"""better outcome""",0,50,0,0
"""unclear""",8,5,5,0
"""null""",17,2,14,7


In [14]:
sup_summary

consensus_w_curator,numerator,denominator,percentage
str,i64,i64,f64
"""poor outcome""",49,52,94.230769
"""better outcome""",50,50,100.0
"""unclear""",5,18,27.777778
"""null""",7,40,17.5


In [15]:
sup_accuracy

0.69375